# SelfSNE quick demo

Minimal Self-Supervised Noise Embedding training on synthetic data.

In [ ]:
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader, TensorDataset
from selfsne import SelfSNE
from selfsne.losses import LikelihoodRatioEstimator

In [ ]:
pl.seed_everything(0)
t = torch.linspace(0, 4*torch.pi, 256)
x = torch.stack([t.sin(), t.cos(), (2*t).sin(), (2*t).cos()], dim=1)
ds = TensorDataset(x[:-1], x[1:])
loader = DataLoader(ds, batch_size=64)

In [ ]:
model = SelfSNE(
    loss=LikelihoodRatioEstimator(divergence="bce"),
    target_encoder=torch.nn.Sequential(
        torch.nn.Linear(4, 32), torch.nn.ReLU(), torch.nn.Linear(32, 2)
    )
)

trainer = pl.Trainer(max_epochs=3, accelerator="cpu", devices=1, logger=False, enable_checkpointing=False, enable_progress_bar=False)
trainer.fit(model, loader)
emb = trainer.predict(model, loader)
embeddings = torch.cat([b["target_embedding"] for b in emb])
print("Embeddings shape:", embeddings.shape)

In [ ]:
# SelfSNE's MI lower bound via likelihood ratio estimation
# The LikelihoodRatioEstimator provides a variational lower bound on I(X;Y)
# We approximate it by the mean log-ratio on held-out pairs
import torch
mi_estimates = []
for xb, yb in loader:
    # use the model to get embeddings
    with torch.no_grad():
        ctx_emb, tgt_emb, _, _ = model.get_embeddings(xb, yb, None)
        logits = model.loss.logits(ctx_emb, tgt_emb)
        # DV lower bound: E[pos] - log E[exp(neg)] approximated by mean pos logit
        mi_estimates.append(logits['pos_logits'].mean().item())
mi_lb = float(np.mean(mi_estimates))
print("SelfSNE MI lower bound estimate:", mi_lb)